# M7 Run 7 - FINAL OOF (M7-v1) of the frozen tuned config

Config = **both (balanced+focal) + strong augmentation + base reg @5000**, 5-seed ensemble. Only change vs Run 2 baseline = strong augmentation. This is the **honest M7-v1 number**.

Full OOF folds 1-8 (checkpointed per fold+seed, mmap cache = memory-safe). fold9 = validation by model reuse. fold10 NEVER touched. Re-measures the ensemble preview (rho, rank-vote) and **re-checks C1** with the tuned model; saves a model for later Grad-CAM.

> Watch: OOF AP vs Run 2 (rich) 0.644 and vs M4 0.718; and the **vote delta** (was +0.004). If standalone rises but vote-delta stays ~0 -> better model, same ensemble conclusion.

### Block 7.0: Setup (frozen config)

In [ ]:
# M7 Run 7 - FINAL OOF (folds 1-8) of the FROZEN tuned config = both + strong aug + base reg @5000.
# = M7-v1 honest number. Re-measures ensemble preview (rho, vote) and re-checks C1 with the tuned model.
# Reuses Run 2 cache (mmap, memory-safe). fold9 = validation by model reuse. fold10 NEVER touched.
import os, sys, json, time, warnings, shutil, zipfile
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score, roc_auc_score
warnings.filterwarnings('ignore')
ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); SRC=os.path.join(ROOT,'src')
FIG=os.path.join(ROOT,'reports','figures'); METRICS=os.path.join(ROOT,'reports','metrics')
MODELS=os.path.join(ROOT,'models','M7_run7'); CACHE_DIR=os.path.join(PROCESSED,'m7_run2_cache')
for d in (FIG,METRICS,MODELS): os.makedirs(d,exist_ok=True)
sys.path.insert(0,SRC)
from evaluation import evaluate_standard, _boot_ci
RANDOM_STATE=42; RESO=5000
N_NEG_TRAIN=12000; SEEDS=[0,1,2,3,4]
EPOCHS=60; PATIENCE=10; BATCH=64; LR=1e-3; WD=1e-4; PDROP=0.3      # base reg
FOCAL_GAMMA=2.0; FOCAL_ALPHA=0.5; VAL_FRAC=0.15; N_JOBS=6
REF={'M3':0.619,'M4':0.718,'vote_M3_M4':0.736}; RUN2_OOF=0.644
C1_OOF=0.65; C1_RHO=0.50
print('M7 Run 7 | FINAL OOF @%d | config both+strong+base | seeds %s | vs Run2(rich) %.3f'%(RESO,SEEDS,RUN2_OOF))
print('Block 7.0 OK.')


### Block 7.1: Reuse Run 2 cache (mmap, memory-safe)

In [ ]:
# Reuse Run 2 cache; X18 memory-mapped (avoid 6 GB contiguous alloc). Small arrays + fold9 loaded normally.
NPZ=os.path.join(CACHE_DIR,'m7_run2_data.npz')
def _mmap_member(name):
    out=os.path.join(CACHE_DIR,name+'.npy')
    if not os.path.exists(out):
        with zipfile.ZipFile(NPZ) as zf, zf.open(name+'.npy') as srcf, open(out,'wb') as dst:
            shutil.copyfileobj(srcf,dst,length=16*1024*1024)
    return np.load(out,mmap_mode='r')
X18=_mmap_member('X18')
z=np.load(NPZ,allow_pickle=True)
eid18=z['eid18']; src18=z['src18']; fold18=z['fold18']; y18=z['y18']; X9=np.asarray(z['X9']); y9=z['y9']; z.close()
print('folds1-8 mmap %s (%d WPW) | fold9 %s (%d WPW)'%(X18.shape,int(y18.sum()),X9.shape,int(y9.sum())))
print('Block 7.1 OK.')


### Block 7.2: 1D-ResNet (identical)

In [ ]:
import torch, torch.nn as nn
torch.set_num_threads(N_JOBS)
class BasicBlock1d(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.conv1=nn.Conv1d(cin,cout,7,stride=stride,padding=3,bias=False); self.bn1=nn.BatchNorm1d(cout)
        self.conv2=nn.Conv1d(cout,cout,7,padding=3,bias=False); self.bn2=nn.BatchNorm1d(cout)
        self.relu=nn.ReLU(inplace=True); self.down=None
        if stride!=1 or cin!=cout:
            self.down=nn.Sequential(nn.Conv1d(cin,cout,1,stride=stride,bias=False),nn.BatchNorm1d(cout))
    def forward(self,x):
        idt=x if self.down is None else self.down(x)
        o=self.relu(self.bn1(self.conv1(x))); o=self.bn2(self.conv2(o)); return self.relu(o+idt)
class ResNet1d(nn.Module):
    def __init__(self,ch=(16,32,64),pdrop=PDROP):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv1d(12,ch[0],15,stride=2,padding=7,bias=False),
                                nn.BatchNorm1d(ch[0]),nn.ReLU(inplace=True),nn.MaxPool1d(4))
        self.layer1=BasicBlock1d(ch[0],ch[0],1); self.layer2=BasicBlock1d(ch[0],ch[1],2); self.layer3=BasicBlock1d(ch[1],ch[2],2)
        self.pool=nn.AdaptiveMaxPool1d(1); self.head=nn.Sequential(nn.Flatten(),nn.Dropout(pdrop),nn.Linear(ch[2],1))
    def forward(self,x):
        return self.head(self.pool(self.layer3(self.layer2(self.layer1(self.stem(x)))))).squeeze(1)
print('Block 7.2 OK | params:', sum(p.numel() for p in ResNet1d().parameters()))


### Block 7.3: Frozen config (strong aug) + training

In [ ]:
# Frozen config: focal loss + STRONG augmentation + balanced batches; early-stop on val loss.
def focal_loss(logits,targets,gamma=FOCAL_GAMMA,alpha=FOCAL_ALPHA):
    ce=nn.functional.binary_cross_entropy_with_logits(logits,targets,reduction='none')
    p=torch.sigmoid(logits); pt=torch.where(targets==1,p,1-p)
    a=torch.where(targets==1,torch.full_like(targets,alpha),torch.full_like(targets,1-alpha))
    return (a*(1-pt).pow(gamma)*ce).mean()
def augment(xb):   # strong
    T=xb.shape[2]; P=dict(shift=0.08,scale=0.3,noise=0.03,wander=0.08,ldrop=0.45)
    sh=max(1,int(P['shift']*T)); xb=torch.roll(xb,int(torch.randint(-sh,sh+1,(1,)).item()),dims=2)
    xb=xb*((1.0-P['scale'])+2*P['scale']*torch.rand(xb.shape[0],1,1)); xb=xb+P['noise']*torch.randn_like(xb)
    t=torch.linspace(0,1,T).view(1,1,-1); fb=0.5+2.0*torch.rand(xb.shape[0],1,1); ph=2*np.pi*torch.rand(xb.shape[0],1,1)
    xb=xb+P['wander']*torch.sin(2*np.pi*fb*t+ph)
    if torch.rand(1).item()<P['ldrop']: xb[:,int(torch.randint(0,12,(1,)).item()),:]=0.0
    if torch.rand(1).item()<0.25: xb[:,int(torch.randint(0,12,(1,)).item()),:]=0.0
    return xb
def predict(model,X):
    model.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(X),256):
            out.append(torch.sigmoid(model(torch.tensor(np.ascontiguousarray(X[i:i+256],dtype=np.float32)))).numpy())
    return np.nan_to_num(np.concatenate(out))
def train_one(seed,Xtr,ytr,Xva,yva):
    torch.manual_seed(seed); np.random.seed(seed); rng=np.random.default_rng(seed)
    pos=np.where(ytr==1)[0]; neg=np.where(ytr==0)[0]; half=BATCH//2
    Xv=torch.tensor(np.ascontiguousarray(Xva,dtype=np.float32)); yv=torch.tensor(yva)
    model=ResNet1d(); opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WD)
    steps=max(1,len(ytr)//BATCH); best=1e9; best_state=None; bad=0
    for ep in range(EPOCHS):
        model.train()
        for _ in range(steps):
            bi=np.concatenate([rng.choice(pos,half,True),rng.choice(neg,half,True)])
            xb=augment(torch.tensor(np.ascontiguousarray(Xtr[bi],dtype=np.float32))); yb=torch.tensor(ytr[bi])
            opt.zero_grad(); loss=focal_loss(model(xb),yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl=[focal_loss(model(Xv[i:i+256]),yv[i:i+256]).item() for i in range(0,len(Xv),256)]
        vloss=float(np.mean(vl))
        if vloss<best-1e-4: best=vloss; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=PATIENCE: break
    model.load_state_dict(best_state); return model
print('Block 7.3 OK.')


### Block 7.4: Final OOF (checkpointed) + save Grad-CAM model

In [ ]:
# OOF folds 1-8 (checkpointed per fold+seed). Saves fold-1/seed-0 model for later Grad-CAM. fold9 by model reuse.
import torch
CKPT=os.path.join(MODELS,'run7_oof_ckpt.npz')
if os.path.exists(CKPT):
    ck=np.load(CKPT,allow_pickle=True); oof=ck['oof'].astype('float32'); f9_acc=ck['f9_acc'].astype('float64')
    f9_n=int(ck['f9_n']); train_ap_folds=list(ck['train_ap_folds']); done_folds=set(int(x) for x in ck['done_folds'])
    print('[ckpt] resumed | folds done %s'%sorted(done_folds))
else:
    oof=np.full(len(y18),np.nan,dtype='float32'); f9_acc=np.zeros(len(y9),dtype='float64'); f9_n=0
    train_ap_folds=[]; done_folds=set()
t_all=time.time()
for f in range(1,9):
    if f in done_folds: print('  fold %d done (ckpt) -> skip'%f); continue
    te_idx=np.where(fold18==f)[0]; pool_idx=np.where(fold18!=f)[0]; rng=np.random.default_rng(1000+f)
    pos=pool_idx[y18[pool_idx]==1]; negall=pool_idx[y18[pool_idx]==0]
    neg=rng.choice(negall,min(N_NEG_TRAIN,len(negall)),replace=False)
    vp=pos.copy(); vn=neg.copy(); rng.shuffle(vp); rng.shuffle(vn)
    nvp=max(6,int(VAL_FRAC*len(vp))); nvn=int(VAL_FRAC*len(vn))
    val_idx=np.concatenate([vp[:nvp],vn[:nvn]]); pool=np.concatenate([pos,neg]); tr_idx=np.setdiff1d(pool,val_idx)
    Xtr=np.ascontiguousarray(X18[tr_idx],dtype=np.float32); ytr=y18[tr_idx]
    Xva=np.ascontiguousarray(X18[val_idx]); yva=y18[val_idx]; Xte=np.ascontiguousarray(X18[te_idx])
    sck=os.path.join(MODELS,'run7_fold%d_seeds.npz'%f)
    if os.path.exists(sck):
        sc=np.load(sck); seed_te=[r for r in sc['te']]; seed_tr=[r for r in sc['tr']]; f9_local=sc['f9'].astype('float64'); done_s=int(sc['done'])
        print('  fold %d: resume from seed %d/%d'%(f,done_s,len(SEEDS)))
    else:
        seed_te=[]; seed_tr=[]; f9_local=np.zeros(len(y9),dtype='float64'); done_s=0
    for si,s in enumerate(SEEDS):
        if si<done_s: continue
        model=train_one(s,Xtr,ytr,Xva,yva)
        if f==1 and si==0: torch.save(model.state_dict(),os.path.join(MODELS,'gradcam_model.pt'))   # for Grad-CAM later
        seed_te.append(predict(model,Xte)); seed_tr.append(predict(model,Xtr)); f9_local=f9_local+predict(model,X9)
        np.savez(sck,te=np.array(seed_te),tr=np.array(seed_tr),f9=f9_local,done=si+1)
    oof[te_idx]=np.mean(seed_te,0); f9_acc=f9_acc+f9_local; f9_n+=len(SEEDS)
    train_ap_folds.append(float(average_precision_score(ytr,np.mean(seed_tr,0)))); done_folds.add(f)
    np.savez(CKPT,oof=oof,f9_acc=f9_acc,f9_n=f9_n,train_ap_folds=np.array(train_ap_folds),done_folds=np.array(sorted(done_folds)))
    if os.path.exists(sck): os.remove(sck)
    done=~np.isnan(oof)
    print('  fold %d done | held-out %d (%d WPW) | OOF-AP so far %.3f | %.1f min'%(
        f,len(te_idx),int(y18[te_idx].sum()),average_precision_score(y18[done],oof[done]),(time.time()-t_all)/60))
    del Xtr,Xva,Xte
f9=(f9_acc/max(f9_n,1)).astype('float32')
np.save(os.path.join(MODELS,'m7v1_oof_folds1_8.npy'),oof); np.save(os.path.join(MODELS,'m7v1_fold9_scores.npy'),f9)
pd.DataFrame({'ecg_id':eid18,'source':src18,'fold':fold18,'label':y18,'m7v1_oof':oof}).to_csv(
    os.path.join(PROCESSED,'m7v1_combined_oof.csv'),index=False)
print('Block 7.4 OK | total %.1f min | models %d'%((time.time()-t_all)/60,f9_n))


### Block 7.5: M7-v1 honest OOF + per-source + standardized fold9

In [ ]:
# Honest M7-v1 OOF metric (folds 1-8) + per-source AP + standardized fold9 eval. Compare to Run 2 (rich) 0.644.
ok=~np.isnan(oof)
OOF_AP=float(average_precision_score(y18[ok],oof[ok])); OOF_AUC=float(roc_auc_score(y18[ok],oof[ok]))
ap_lo,ap_hi=_boot_ci(y18[ok].astype(int),oof[ok],average_precision_score)
ap_train=float(np.mean(train_ap_folds)); gap=ap_train-OOF_AP
print('=== M7-v1 (Run 7) OOF (folds 1-8, %d WPW) ==='%int(y18[ok].sum()))
print('  OOF AP  %.3f CI95[%.3f,%.3f]   Run2(rich) %.3f | M3 %.3f | M4 %.3f | vote %.3f'%(OOF_AP,ap_lo,ap_hi,RUN2_OOF,REF['M3'],REF['M4'],REF['vote_M3_M4']))
print('  OOF AUC %.3f | gap train-OOF %+.3f (train %.3f) | delta vs Run2 %+.3f'%(OOF_AUC,gap,ap_train,OOF_AP-RUN2_OOF))
for s in ['ptb','ningbo']:
    m=ok & np.array([str(x).lower().startswith(s) for x in src18])
    if m.sum() and y18[m].sum()>=2:
        print('   OOF AP [%-6s] %.3f (%d WPW / %d ECG)'%(s,float(average_precision_score(y18[m],oof[m])),int(y18[m].sum()),int(m.sum())))
evaluate_standard(name='M7v1_oof_5000',y_oof=y18[ok],score_oof=oof[ok],y_test=y9,score_test=f9,
                  figures_dir=FIG,metrics_dir=METRICS,ap_train=ap_train,
                  extra={'run':'m7_run7','config':'both+strong+base','OOF_AP':OOF_AP,'OOF_AP_CI':[ap_lo,ap_hi],'OOF_AUC':OOF_AUC,'gap':gap})
print('Block 7.5 OK.')


### Block 7.6: Ensemble preview re-measured (read-only)

In [ ]:
# READ-ONLY ensemble preview re-measured with the tuned M7-v1. Align by (ecg_id, source) with M3/M4 OOF.
m3=pd.read_csv(os.path.join(PROCESSED,'m3_combined_oof.csv'),dtype={'ecg_id':str})[['ecg_id','source','proba_raw']].rename(columns={'proba_raw':'m3'})
m4=pd.read_csv(os.path.join(PROCESSED,'m4_combined_oof.csv'),dtype={'ecg_id':str})[['ecg_id','source','proba_raw']].rename(columns={'proba_raw':'m4'})
m7=pd.DataFrame({'ecg_id':np.asarray(eid18,dtype=str),'source':np.asarray(src18,dtype=str),'m7':oof,'label':y18}); m7=m7[~np.isnan(m7.m7)]
D=m7.merge(m3,on=['ecg_id','source']).merge(m4,on=['ecg_id','source'])
def rk(a): return pd.Series(np.asarray(a)).rank(pct=True).values
rho_m3=float(spearmanr(D.m7,D.m3).correlation); rho_m4=float(spearmanr(D.m7,D.m4).correlation)
vote2=rk(D.m3)+rk(D.m4); vote3=rk(D.m3)+rk(D.m4)+rk(D.m7)
ap2=float(average_precision_score(D.label,vote2)); ap3=float(average_precision_score(D.label,vote3))
print('aligned %d (%d WPW) | rho(M7,M3) %.3f | rho(M7,M4) %.3f'%(len(D),int(D.label.sum()),rho_m3,rho_m4))
print('  rank-vote AP : M3+M4 %.3f -> M3+M4+M7 %.3f   (delta %+.3f)   [Run2 was +0.004]'%(ap2,ap3,ap3-ap2))
json.dump({'OOF_AP':OOF_AP,'rho_M3':rho_m3,'rho_M4':rho_m4,'rankvote_M3M4':ap2,'rankvote_M3M4M7':ap3,'delta':ap3-ap2},
          open(os.path.join(METRICS,'m7_run7_ensemble_preview.json'),'w'),indent=2)
print('Block 7.6 OK.')


### Block 7.7: Gate C1 re-check (informational)

In [ ]:
# C1 RE-CHECK with the tuned M7-v1 (informational; C1 was FAIL at Run 2 = 0.644). Any override journaled.
rho_max=float(max(rho_m3,rho_m4)); pass_oof=bool(OOF_AP>=C1_OOF); pass_rho=bool(rho_max<C1_RHO); C1=bool(pass_oof and pass_rho)
print('=== GATE C1 RE-CHECK (tuned M7-v1) ===')
print('  OOF AP %.3f (>=%.2f ? %s) | max rho %.3f (<%.2f ? %s)'%(OOF_AP,C1_OOF,pass_oof,rho_max,C1_RHO,pass_rho))
print('  C1: %s'%('PASS' if C1 else 'FAIL'))
print('  Reading: if C1 now PASSES but vote-delta still ~0 -> standalone improved, ensemble unchanged (rho necessary not sufficient).')
print('  Pretraining (Runs 8-10) decision: default policy (stay closed) unless C1 PASS *and* a real ensemble case; journal the decision.')
json.dump({'OOF_AP':OOF_AP,'rho_max':rho_max,'C1_pass':C1,'vote_delta':ap3-ap2,'delta_vs_run2':OOF_AP-RUN2_OOF},
          open(os.path.join(METRICS,'m7_run7_C1_recheck.json'),'w'),indent=2)
print('Block 7.7 OK.')
